# Self-preference and verbosity bias [Step 07.03 - The biases swapping does not fix]

> **MLCourse - Agentic AI - Agent Patterns**

Position bias is fixable by swapping. These two are not, because they are about the
*content*, and swapping preserves content exactly.

- **Verbosity bias** - longer answers score higher, holding correctness constant.
  This is the best-documented judge bias after position, and it is dangerous because
  it creates an *optimisation target*: a system tuned against a verbose-biased judge
  learns to pad.
- **Self-preference bias** - a judge prefers text produced by itself (or by its own
  model family) over equally good text from elsewhere. It matters most in exactly
  the setup people reach for first: using the same model to generate and to grade.

### Key takeaways

- Test verbosity with **content-matched pairs**: the same facts, one short, one
  padded. Any preference is bias by construction.
- Test self-preference by having the judge grade its own output against a
  different-provenance output of matched quality.
- The mitigation for verbosity is a rubric that **penalises unsupported additions**
  and a length cap in the generation prompt - not a plea to "ignore length".
- The mitigation for self-preference is a **different judge model**, or humans.

### Setup: environment, model factory, rate-limit-aware call helper


In [ ]:
import os                                   # environment variables
import time                                 # timing + backoff sleeps
from pathlib import Path                    # locating the .env
from dotenv import load_dotenv              # reads KEY=value pairs from .env

# Walk UP from this notebook until we find the folder that CONTAINS the track
# directory `03_agentic_ai` (that folder is the repo root), then load the
# gitignored .env that lives INSIDE the track.
#
# Pitfall worth naming: it is easy to write the walk so that it stops at the
# repo root and then load `ROOT/.env`, which does not exist - `load_dotenv`
# returns False and says nothing, so the notebook silently has no key.
ROOT = Path.cwd()
while not (ROOT / "03_agentic_ai").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ENV_PATH = ROOT / "03_agentic_ai" / ".env"
load_dotenv(ENV_PATH)

GROQ_MODEL = "qwen/qwen3.8-27b"             # the one hosted model this course uses
GROQ_KEY = os.environ["GROQ_API_KEY"]       # KeyError here = .env not found. Never print it.

# A local Ollama model (e.g. `llama3.1:8b`) is a perfectly good substitute if you
# have no Groq key - swap the two lines in `make_llm`. We deliberately do NOT
# write a silent fallback branch: a notebook that quietly changes model behind
# your back produces numbers you cannot trust.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 256):
    """Return the chat model used everywhere in this module."""
    return ChatGroq(model=GROQ_MODEL, api_key=GROQ_KEY,
                    temperature=temperature, max_tokens=max_tokens)


PACE = 0.7          # seconds to wait between calls: the free tier is 8000 TPM


def safe_invoke(model, messages, retries: int = 5, pause: float = 2.0):
    """Invoke a chat model, backing off exponentially on 429 / rate-limit errors.

    Returns the AIMessage. Raises if every retry is exhausted - we want a loud
    failure, not a quiet wrong number.
    """
    for attempt in range(retries):
        try:
            out = model.invoke(messages)
            time.sleep(PACE)                       # pace the next call
            return out
        except Exception as exc:                   # noqa: BLE001 - we re-raise below
            text = str(exc).lower()
            if "429" in text or "rate" in text or "quota" in text:
                wait = pause * (2 ** attempt)
                print("  [rate limit] sleeping %.1fs (attempt %d/%d)" % (wait, attempt + 1, retries))
                time.sleep(wait)
                continue
            raise
    raise RuntimeError("rate limited after %d attempts" % retries)


# Published Groq list price for this model at the time of writing, in USD per
# 1M tokens. Substitute your own numbers - the METHOD is the lesson, not these
# two constants.
PRICE_IN_PER_M = 0.29
PRICE_OUT_PER_M = 0.59


def usd(in_tok: int, out_tok: int) -> float:
    """Convert a token count into dollars at the prices above."""
    return in_tok / 1e6 * PRICE_IN_PER_M + out_tok / 1e6 * PRICE_OUT_PER_M


print("env file :", ENV_PATH, "(exists:", ENV_PATH.exists(), ")")
print("model    :", GROQ_MODEL)
print("key      : loaded, %d chars" % len(GROQ_KEY))


In [2]:
PACE = 2.0
print("PACE =", PACE)

PACE = 2.0


### The evaluation pairs


In [ ]:
# Five questions, each with a GOOD answer and a WORSE answer. "Worse" is worse for
# a stated, checkable reason - not just shorter or blander - so we know the right
# verdict without asking anyone.
#
# Building the pairs by hand like this is the only way to measure a judge: you need
# ground truth ABOUT THE JUDGE, which means you must know the answer already.

PAIRS = [
    dict(
        id="photosynthesis",
        q="In one or two sentences, what does photosynthesis produce?",
        good="Photosynthesis produces glucose and oxygen, using carbon dioxide, water and light energy.",
        bad="Photosynthesis produces carbon dioxide and water, which the plant then releases into the air.",
        why_bad="reverses the reactants and the products - factually wrong",
    ),
    dict(
        id="http_status",
        q="What does HTTP status code 404 mean?",
        good="404 means the server understood the request but could not find the requested resource.",
        bad="404 means the server is temporarily overloaded and the client should retry later.",
        why_bad="describes 503, not 404",
    ),
    dict(
        id="python_list",
        q="What is the time complexity of appending to a Python list?",
        good="Amortised O(1): appends are constant time on average, with occasional O(n) reallocations.",
        bad="O(n), because the list has to be copied every time an element is added.",
        why_bad="wrong complexity - ignores the amortised growth strategy",
    ),
    dict(
        id="vaccine",
        q="Briefly, how do mRNA vaccines work?",
        good=("They deliver mRNA instructing your cells to make a harmless viral protein, "
              "which the immune system then learns to recognise."),
        bad=("They inject a weakened live virus that reproduces slowly so the immune system "
             "can practise fighting it."),
        why_bad="describes a live attenuated vaccine, not an mRNA vaccine",
    ),
    dict(
        id="git_rebase",
        q="What does `git rebase` do, in one sentence?",
        good="It replays your commits on top of another base commit, rewriting their history.",
        bad="It merges two branches together and creates a merge commit recording both parents.",
        why_bad="describes merge, which is the thing rebase is contrasted with",
    ),
]

print("%d pairs" % len(PAIRS))
for p in PAIRS:
    print("  %-16s bad answer: %s" % (p["id"], p["why_bad"]))


### A pairwise judge


In [ ]:
# Reply format is fixed so a REGEX reads the verdict. Never parse a judge's verdict
# with another LLM: you would then need to evaluate that one too.

import re

judge_llm = make_llm(temperature=0.0, max_tokens=200)

PAIRWISE_SYSTEM = (
    "You are an impartial evaluator. You will see a question and two candidate "
    "answers, A and B. Decide which answer is more FACTUALLY CORRECT. "
    "Ignore length, tone, formatting and confidence. "
    "Reply with one short sentence of justification, then a final line of exactly "
    "the form 'VERDICT: A' or 'VERDICT: B'.")

VERDICT_RE = re.compile(r"VERDICT:\s*([AB])")


def judge_pair(question, answer_a, answer_b):
    """Return ('A'|'B'|'?', raw_text)."""
    m = safe_invoke(judge_llm, [
        ("system", PAIRWISE_SYSTEM),
        ("user", "QUESTION:\n%s\n\nANSWER A:\n%s\n\nANSWER B:\n%s"
                 % (question, answer_a, answer_b))])
    v = VERDICT_RE.search(m.content)
    return (v.group(1) if v else "?"), m.content


### 1. Verbosity bias, measured

Content-matched pairs: the long version states the **same facts** as the short one,
padded with true but unrequested material. Neither is more correct. If the judge
prefers the long one systematically, that is bias, not judgement.

Note we still swap the order, so position bias cannot contaminate the result.

In [5]:
VERBOSITY = [
    dict(id="tcp",
         q="What is the main difference between TCP and UDP?",
         short="TCP is connection-oriented and guarantees ordered, reliable delivery. UDP is connectionless and does neither.",
         long=("TCP, the Transmission Control Protocol, is a connection-oriented protocol, which "
               "means that before any data is exchanged the two endpoints perform a handshake to "
               "establish a connection. Once established, TCP guarantees that the bytes you send "
               "arrive, that they arrive exactly once, and that they arrive in the order you sent "
               "them, using sequence numbers, acknowledgements and retransmission. UDP, the User "
               "Datagram Protocol, is connectionless: there is no handshake, no acknowledgement, "
               "no retransmission and no ordering guarantee. Each datagram is independent.")),
    dict(id="index",
         q="Why does a database index speed up queries?",
         short="An index is a sorted structure that lets the engine find matching rows without scanning the whole table.",
         long=("A database index is an auxiliary data structure, most commonly a B-tree, that stores "
               "the indexed column values in sorted order along with pointers back to the rows they "
               "came from. Because the values are sorted, the database engine can locate a matching "
               "value by descending the tree rather than by examining every row in the table. This "
               "turns what would be a full table scan, proportional to the size of the table, into a "
               "lookup proportional to the logarithm of that size.")),
    dict(id="cache",
         q="What is a cache hit ratio?",
         short="The fraction of requests served from the cache rather than the underlying store.",
         long=("The cache hit ratio is a metric describing how effectively a cache is doing its job. "
               "It is defined as the number of requests that the cache was able to satisfy from its "
               "own contents, divided by the total number of requests it received, usually expressed "
               "as a percentage. The complement of the hit ratio is the miss ratio. A higher hit "
               "ratio means fewer requests reach the slower backing store.")),
]

for v in VERBOSITY:
    v["len_short"] = len(v["short"].split())
    v["len_long"] = len(v["long"].split())
    print("%-8s short %3d words, long %3d words (%.1fx)"
          % (v["id"], v["len_short"], v["len_long"], v["len_long"] / v["len_short"]))

tcp      short  14 words, long  81 words (5.8x)
index    short  18 words, long  83 words (4.6x)
cache    short  13 words, long  71 words (5.5x)


In [6]:
long_wins = ties = short_wins = 0
for v in VERBOSITY:
    a, _ = judge_pair(v["q"], v["short"], v["long"])     # order 1: short is A
    b, _ = judge_pair(v["q"], v["long"], v["short"])     # order 2: long is A
    pick_1 = "short" if a == "A" else "long"
    pick_2 = "long" if b == "A" else "short"
    if pick_1 == pick_2 == "long":
        long_wins += 1
        outcome = "LONG"
    elif pick_1 == pick_2 == "short":
        short_wins += 1
        outcome = "short"
    else:
        ties += 1
        outcome = "tie (judge flipped)"
    print("%-8s order1->%-6s order2->%-6s  => %s" % (v["id"], pick_1, pick_2, outcome))

nv = len(VERBOSITY)
print()
print("=" * 58)
print("VERBOSITY BIAS (n=%d content-matched pairs, %d calls)" % (nv, 2 * nv))
print("-" * 58)
print("long preferred  : %d/%d = %.0f%%" % (long_wins, nv, 100 * long_wins / nv))
print("short preferred : %d/%d = %.0f%%" % (short_wins, nv, 100 * short_wins / nv))
print("tie / flipped   : %d/%d" % (ties, nv))
print("=" * 58)
print("Chance is 33/33/33. Anything far above 33%% for LONG is verbosity bias,")
print("because the two answers state the same facts by construction.")

tcp      order1->long   order2->long    => LONG


index    order1->long   order2->long    => LONG


cache    order1->long   order2->short   => tie (judge flipped)

VERBOSITY BIAS (n=3 content-matched pairs, 6 calls)
----------------------------------------------------------
long preferred  : 2/3 = 67%
short preferred : 0/3 = 0%
tie / flipped   : 1/3
Chance is 33/33/33. Anything far above 33%% for LONG is verbosity bias,
because the two answers state the same facts by construction.


### The mitigation that actually works

Telling the judge to "ignore length" barely helps - the pairwise prompt in this
module already says exactly that, and you can see above what it bought.

What does help is changing *what is being scored*:

- Score a named dimension (**correctness**, **groundedness**) rather than
  "which is better".
- Add an explicit penalty clause: *"Unrequested material that does not answer the
  question does not improve the score and may lower it."*
- Cap generation length in the **generator's** prompt, so the candidates are
  length-matched before the judge ever sees them. This is the most reliable fix, and
  it is not the judge's job at all.

Let's try the penalty clause and see whether it moves the number.

In [7]:
STRICT_SYSTEM = (
    "You are an impartial evaluator. You will see a question and two candidate "
    "answers, A and B. Decide which answer better ANSWERS THE QUESTION ASKED. "
    "Unrequested background material does not improve an answer and may make it "
    "worse by burying the point. Length is not a merit. "
    "If both answers state the same facts, prefer the more direct one. "
    "Reply with one short sentence, then a final line 'VERDICT: A' or 'VERDICT: B'.")

strict_llm = make_llm(temperature=0.0, max_tokens=200)


def judge_pair_strict(question, a, b):
    m = safe_invoke(strict_llm, [("system", STRICT_SYSTEM),
                                 ("user", "QUESTION:\n%s\n\nANSWER A:\n%s\n\nANSWER B:\n%s"
                                          % (question, a, b))])
    v = VERDICT_RE.search(m.content)
    return (v.group(1) if v else "?")


s_long = s_short = s_tie = 0
for v in VERBOSITY:
    a = judge_pair_strict(v["q"], v["short"], v["long"])
    b = judge_pair_strict(v["q"], v["long"], v["short"])
    p1 = "short" if a == "A" else "long"
    p2 = "long" if b == "A" else "short"
    if p1 == p2 == "long":
        s_long += 1
    elif p1 == p2 == "short":
        s_short += 1
    else:
        s_tie += 1
    print("%-8s order1->%-6s order2->%-6s" % (v["id"], p1, p2))

print()
print("%-28s %-10s %-10s %-10s" % ("rubric", "long", "short", "tie"))
print("-" * 60)
print("%-28s %-10d %-10d %-10d" % ("generic 'more correct'", long_wins, short_wins, ties))
print("%-28s %-10d %-10d %-10d" % ("with anti-padding clause", s_long, s_short, s_tie))
print("-" * 60)

tcp      order1->short  order2->short 


index    order1->long   order2->long  


cache    order1->long   order2->short 

rubric                       long       short      tie       
------------------------------------------------------------
generic 'more correct'       2          0          1         
with anti-padding clause     1          1          1         
------------------------------------------------------------


### 2. Self-preference bias

The clean experiment: have the model **write** an answer, then have the *same model*
grade its own answer against a human-written answer of comparable quality. Any
systematic preference for its own text is self-preference.

The confound to control for is real quality difference, so we do two things:
we swap positions (killing position bias), and we use questions where a short
human-written answer is genuinely correct and complete.

In [8]:
SELF_Q = [
    ("What is a race condition in concurrent programming?",
     "A race condition is when the result of a program depends on the unpredictable "
     "order in which two or more threads reach a shared piece of state."),
    ("What does 'idempotent' mean for an HTTP method?",
     "An idempotent method can be called many times with the same effect as calling "
     "it once, so retrying it is safe."),
    ("What is the purpose of a foreign key?",
     "A foreign key constrains a column to values that exist in another table's key, "
     "so references between tables stay valid."),
]

writer = make_llm(temperature=0.0, max_tokens=120)
own_answers = []
for q, human in SELF_Q:
    m = safe_invoke(writer, [("system", "Answer in one or two sentences. Be precise."),
                             ("user", q)])
    own_answers.append(m.content.strip())
    print("Q:", q)
    print("  human-written :", human)
    print("  model-written :", own_answers[-1].replace("\n", " ")[:180])
    print()

Q: What is a race condition in concurrent programming?
  human-written : A race condition is when the result of a program depends on the unpredictable order in which two or more threads reach a shared piece of state.
  model-written : A race condition is a logical error in concurrent programming where the output of a program depends on the unpredictable timing or interleaving of multiple threads accessing shared



Q: What does 'idempotent' mean for an HTTP method?
  human-written : An idempotent method can be called many times with the same effect as calling it once, so retrying it is safe.
  model-written : An HTTP method is idempotent if making the same request multiple times has the same effect on the server state as making it once. This means that repeated executions of the request



Q: What is the purpose of a foreign key?
  human-written : A foreign key constrains a column to values that exist in another table's key, so references between tables stay valid.
  model-written : A foreign key is a column or set of columns in a relational database table that establishes and enforces a link between the data in two tables by referencing the primary key of ano



In [9]:
own_wins = human_wins = flip = 0
for (q, human), own in zip(SELF_Q, own_answers):
    a, _ = judge_pair(q, own, human)        # order 1: own is A
    b, _ = judge_pair(q, human, own)        # order 2: own is B
    p1 = "own" if a == "A" else "human"
    p2 = "own" if b == "B" else "human"
    if p1 == p2 == "own":
        own_wins += 1
        r = "OWN"
    elif p1 == p2 == "human":
        human_wins += 1
        r = "human"
    else:
        flip += 1
        r = "tie (flipped)"
    print("%-56s -> %s" % (q[:56], r))

ns = len(SELF_Q)
print()
print("=" * 58)
print("SELF-PREFERENCE (n=%d, %d judge calls, positions swapped)" % (ns, 2 * ns))
print("-" * 58)
print("preferred its OWN answer   : %d/%d = %.0f%%" % (own_wins, ns, 100 * own_wins / ns))
print("preferred the human answer : %d/%d = %.0f%%" % (human_wins, ns, 100 * human_wins / ns))
print("inconsistent (tie)         : %d/%d" % (flip, ns))
print("=" * 58)
print()
print("Caveats you must state with a number like this:")
print(" - n=3. The confidence interval is enormous. This is a PROCEDURE, not a fact.")
print(" - The model's answers may genuinely be better; 'self-preference' is only")
print("   established when quality is matched, which needs human labels to confirm.")
print(" - Same model, same prompt style: the effect here is the WEAKEST version.")
print("   The real risk is grading your own pipeline's output with your own model.")

What is a race condition in concurrent programming?      -> tie (flipped)


What does 'idempotent' mean for an HTTP method?          -> OWN


What is the purpose of a foreign key?                    -> tie (flipped)

SELF-PREFERENCE (n=3, 6 judge calls, positions swapped)
----------------------------------------------------------
preferred its OWN answer   : 1/3 = 33%
preferred the human answer : 0/3 = 0%
inconsistent (tie)         : 2/3

Caveats you must state with a number like this:
 - n=3. The confidence interval is enormous. This is a PROCEDURE, not a fact.
 - The model's answers may genuinely be better; 'self-preference' is only
   established when quality is matched, which needs human labels to confirm.
 - Same model, same prompt style: the effect here is the WEAKEST version.
   The real risk is grading your own pipeline's output with your own model.


### 3. What to do about it

| Bias | Survives swapping? | Practical fix |
|---|---|---|
| Position | No | Swap and require agreement; record ties |
| Verbosity | Yes | Score a named dimension; anti-padding clause; **cap generation length** |
| Self-preference | Yes | **Use a different model family as judge**; or humans on a sample |
| Score clustering | Yes | Anchored rubric; pairwise instead of pointwise |

The uncomfortable conclusion: **do not grade your own model's output with your own
model** if the result will influence a decision you care about. If you have no
second model, at minimum calibrate against human labels on a sample - which is
exactly what notebook 04 does.

### Pitfalls

- **Concluding "no bias" from n=3.** Say the sample size next to the number, every
  time.
- **Quality confounds.** If the two answers differ in quality, you are measuring
  quality, not bias. Content-matched pairs are the only clean design.
- **Fixing the judge instead of the generator.** Verbosity bias is often cheapest to
  fix by making the generator terser.

### Next

Notebook 04 asks the only question that ultimately validates a judge: **does it
agree with humans?** We calibrate against the human labels collected in
`../../03_rag_advanced/10_rag_evaluation/03_human_evaluation.ipynb`.